# Project 1 figures
Run the first two code cells, then the required figure cells. PDFs are saved here and copied to the thesis.

In [1]:
from pathlib import Path
import sys, shutil, json, re, hashlib
sys.dont_write_bytecode = True
HERE = Path.cwd()
if HERE.name != "figures_for_thesis": HERE = HERE / "figures_for_thesis"
ROOT = HERE.parent
THESIS = ROOT.parent / "PhD_thesis_20251216" / "figures_proj1"
sys.path.insert(0, str(HERE))
sys.path.insert(0, str(ROOT))
import numpy as np
import h5py
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator, FormatStrFormatter
from scipy.interpolate import griddata
from PIL import Image
from IPython.display import display
from vmatplot.algorithms import fit_eos
import electronic, optics
from electronic import bands, dos, BAND_COLORS, PDOS_COLORS
from optics import dielectric, BILAYERS, MONOLAYERS, HSE, COMPONENTS, NAMES

plt.rcParams.update({
    "font.family": "serif", "mathtext.fontset": "cm", "font.size": 14,
    "axes.labelsize": 16, "axes.titlesize": 20, "xtick.labelsize": 14,
    "ytick.labelsize": 14, "legend.fontsize": 14, "figure.dpi": 196,
    "lines.linewidth": 1.5, "lines.solid_capstyle": "round",
    "lines.dash_capstyle": "round", "lines.solid_joinstyle": "round",
    "lines.dash_joinstyle": "round", "xtick.direction": "in",
    "ytick.direction": "in", "xtick.top": True, "ytick.right": True,
    "pdf.fonttype": 42, "path.simplify": False,
})
BLUE, GREEN, VIOLET, GREY, ORANGE = "#1478E1", "#28AF3C", "#8C64E1", "#787878", "#FA8C00"
FERMI = "#643CC3"
TAB = {"boxstyle": "round", "facecolor": "white",
       "edgecolor": plt.rcParams["legend.edgecolor"],
       "alpha": plt.rcParams["legend.framealpha"]}



def title(ax, text):
    ax.set_title(text, loc="left", x=0.045, y=0.955, pad=0, va="top",
                 fontsize=12, bbox=TAB, zorder=10)


def grid(rows, cols, height, right_legend=False):
    fig, axes = plt.subplots(rows, cols, figsize=(10, height), squeeze=False)
    fig.subplots_adjust(left=0.10, right=0.75 if right_legend else 0.98,
                        bottom=0.12 if right_legend else 0.21, top=0.97,
                        wspace=0.27, hspace=0.25)
    return fig, axes


def legend(fig, handles, labels=None, right=False, columns=2):
    if right:
        fig.legend(handles, labels, loc="center left", bbox_to_anchor=(0.77, 0.59),
                   frameon=True, fancybox=True)
    else:
        fig.legend(handles, labels, loc="lower left", bbox_to_anchor=(0.09, 0.005),
                   ncol=columns, frameon=True, fancybox=True)



GENERATED = set()
def save(fig, name):
    fig.savefig(HERE / name, metadata={"CreationDate": None})
    shutil.copyfile(HERE / name, THESIS / name)
    GENERATED.add(name)
    display(fig)
    plt.close(fig)

In [2]:
SOURCE = ROOT / "2_Structure_and_CDD"
manifest = {}


def panel(ax, filename, heading):
    path = SOURCE / filename
    pixels = np.asarray(Image.open(path))
    ax.imshow(pixels, interpolation="none")
    ax.set_xticks([])
    ax.set_yticks([])
    title(ax, heading)
    manifest[filename] = {
        "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
        "size": list(pixels.shape[:2][::-1]),
    }
    return pixels.shape[1], pixels.shape[0]

def draw_bands(ax, prefix):
    for i, functional in enumerate(["PBE", "HSE"]):
        x, energy, ticks, conduction, valence = bands(prefix + functional)
        for mask, color in [(conduction, BAND_COLORS[2*i]), (valence, BAND_COLORS[2*i+1])]:
            ax.plot(x, energy[mask].T, color=color)
    ax.set(xlim=(0, 1), ylim=(-6, 6), yticks=np.arange(-6, 7, 2),
           xticks=ticks, xticklabels=[r"$\Gamma$", "K", "M", r"$\Gamma$"])
    for x in ticks[1:-1]: ax.axvline(x, color=GREY, ls="--", zorder=0)
    ax.axhline(0, color=FERMI, ls="--", zorder=0)


BAND_HANDLES = [Line2D([], [], color=c) for c in BAND_COLORS] + [Line2D([], [], color=FERMI, ls="--")]
BAND_LABELS = ["Conduction\n(PBE)", "Valence\n(PBE)", "Conduction\n(HSE06)", "Valence\n(HSE06)", "Fermi energy"]

ENERGY_SOURCES = {}

def table(path):
    data = np.loadtxt(ROOT / path, skiprows=1)
    ENERGY_SOURCES[path] = {"columns": (ROOT / path).read_text().splitlines()[0], "shape": list(data.shape), "energy_column": -1}
    return data

def spectrum(ax, systems, direction, quantity):
    for label, source, color in systems:
        x, y = optics.values(source, direction, quantity)
        ax.plot(x, y, color=color, label=label)
    ax.set(xlim=(0,25), xticks=[0,5,10,15,20,25], ylim=(0,None))
    low,high=ax.get_ylim();ax.set_ylim(low,high+.18*(high-low))


def optical(name, quantities, systems, directions, height):
    fig, axes = grid(len(quantities), len(directions), height)
    fig.subplots_adjust(bottom=.16 if len(quantities) > 2 else .25,
                        hspace=.30, wspace=.30)
    for row, (quantity, ylabel) in enumerate(quantities):
        for col, direction in enumerate(directions):
            ax = axes[row,col]
            spectrum(ax, systems, direction, quantity)
            title(ax, f"({chr(97+row*len(directions)+col)}) " + NAMES[direction])
            if col == 0: ax.set_ylabel(ylabel)
            if row == len(quantities)-1: ax.set_xlabel("Photon energy (eV)")
            else: ax.tick_params(labelbottom=False)
    handles, labels = axes[0,0].get_legend_handles_labels()
    legend(fig, handles, labels, columns=2)
    save(fig, name)

### proj1.1 — Monolayer structures

In [3]:
# The seven bond markers retain the coordinates and colours in the original
# 0.1_structure_figure.ipynb. Coordinates there refer to a 2 x 2 composite.
fig, axes = plt.subplots(2, 2, figsize=(10, 6.0))
fig.subplots_adjust(left=0.01, right=0.99, bottom=0.015, top=0.985,
                    wspace=0.025, hspace=0.035)
names = ["A_BC3.png", "B_Borophene.png", "C_B4C3.png", "D_Graphene.png"]
headings = [r"(a) BC$_3$", "(b) Borophene", r"(c) B$_4$C$_3$", "(d) Graphene"]
for ax, name, heading in zip(axes.flat, names, headings):
    width, height = panel(ax, name, heading)

shift_x, shift_y = 0.008 * 0.866, 0.008 * 0.5
markers = [
    (1, (0, 1), (3772, 654), (4052, 655), (0, -0.008), (0, -0.035), "#145AAA"),
    (2, (0, 1), (4654, 653), (4931, 653), (0, -0.008), (0, -0.035), "#238C4B"),
    (3, (0, 1), (4414, 1010), (4556, 763), (shift_x, shift_y), (0.035, 0.060), "#643CC3"),
    (4, (1, 0), (1647, 2403), (1785, 2163), (shift_x, shift_y), (0.035, 0.060), "#787D8C"),
    (5, (1, 0), (820, 2379), (968, 2161), (shift_x, shift_y), (0.035, 0.060), "#EB731E"),
    (6, (1, 0), (1396, 2730), (1518, 2518), (-shift_x, -shift_y), (-0.025, -0.040), "#AA1E64"),
    (7, (1, 0), (1241, 3104), (1364, 2892), (shift_x, shift_y), (0.035, 0.060), "#AA3CB9"),
]
for number, (row, col), start, end, offset, label_offset, colour in markers:
    scale = np.array([width, height])
    origin = np.array([col, row])
    start = np.array(start) / [2976, 1749] - origin
    end = np.array(end) / [2976, 1749] - origin
    ax = axes[row, col]
    ax.annotate("", xy=(start + offset) * scale, xytext=(end + offset) * scale,
                arrowprops=dict(arrowstyle="<->", color=colour, lw=1.5,
                                shrinkA=0, shrinkB=0, mutation_scale=8))
    position = ((start + end) / 2 + label_offset) * scale
    ax.text(*position, rf"$l_{number}$", fontsize=16, ha="center", color=colour)
save(fig, "proj1.1.pdf")

### proj1.3 — Bilayer structures

In [4]:
fig, axes = plt.subplots(1, 3, figsize=(10, 2.05))
fig.subplots_adjust(left=0.01, right=0.99, bottom=0.03, top=0.97, wspace=0.025)
materials = [r"Graphene–BC$_3$", "Graphene–Borophene", r"Graphene–B$_4$C$_3$"]
prefixes = ["E_Graphene-BC3", "F_Graphene-Borophene", "G_Graphene-B4C3"]
for ax, prefix, material, letter in zip(axes, prefixes, materials, "abc"):
    panel(ax, prefix + ".png", f"({letter}) {material}")
save(fig, "proj1.3.pdf")

### proj1.4 — Charge-density differences

In [5]:
materials = [r"Graphene–BC$_3$", "Graphene–Borophene", r"Graphene–B$_4$C$_3$"]
prefixes = ["E_Graphene-BC3", "F_Graphene-Borophene", "G_Graphene-B4C3"]
fig, axes = plt.subplots(3, 3, figsize=(10, 6.1))
fig.subplots_adjust(left=0.01, right=0.99, bottom=0.015, top=0.985,
                    wspace=0.025, hspace=0.04)
for col, (prefix, material) in enumerate(zip(prefixes, materials)):
    for row, (suffix, view) in enumerate(zip(
            ["top1", "side1a", "bottom1"], ["Top view", "Side view", "Bottom view"])):
        heading = f"({chr(97 + col)}) {material}" if row == 0 else view
        panel(axes[row, col], prefix + "_" + suffix + ".png", heading)
save(fig, "proj1.4.pdf")

### proj1.3_bands — Monolayer bands

In [6]:
handles, labels = BAND_HANDLES, BAND_LABELS
fig, axes = grid(2, 2, 6.4, right_legend=True)
for i, (ax, prefix, label) in enumerate(zip(axes.flat,
        ["D_Graphene_", "B_Borophene_", "A_BC3_", "C_B4C3_"],
        ["Graphene", "Borophene", r"BC$_3$", r"B$_4$C$_3$"])):
    draw_bands(ax, prefix)
    title(ax, f"({chr(97+i)}) " + label)
    if i % 2 == 0: ax.set_ylabel("Energy (eV)")
    else: ax.tick_params(labelleft=False)
    if i >= 2: ax.set_xlabel(r"Wave vector ($k$)")
legend(fig, handles, labels, right=True)
save(fig, "proj1.3_bands.pdf")

### proj1.5–7 — Bilayer bands and DoS

In [7]:
handles, labels = BAND_HANDLES, BAND_LABELS
for name, prefix, dprefix, label in [
    ("proj1.5.pdf", "E_Graphene-BC3_hollow_", "E_Graphene-BC3_", r"Graphene-BC$_3$ (Hollow)"),
    ("proj1.6.pdf", "F_Graphene-Borophene_top_", "F_Graphene-Borophene_", "Graphene-Borophene (Top)"),
    ("proj1.7.pdf", "G_Graphene-B4C3_top_", "G_Graphene-B4C3_", r"Graphene-B$_4$C$_3$ (Top)"),
]:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.8), sharey=True,
                             gridspec_kw={"width_ratios": [3, 1]})
    fig.subplots_adjust(left=.09, right=.72, bottom=.15, top=.90, wspace=.15)
    draw_bands(axes[0], prefix)
    for i, functional in enumerate(["PBE", "HSE"]):
        folder = dprefix + functional + ("_K33" if prefix.startswith("G_") and i == 0 else "")
        energy, total, _, _ = dos(folder)
        for mask, color in [(energy > 0, BAND_COLORS[2*i]), (energy < 0, BAND_COLORS[2*i+1])]:
            axes[1].plot(total[mask], energy[mask], color=color)
    axes[0].set(xlabel=r"Wave vector ($k$)", ylabel="Energy (eV)")
    axes[1].set(xlim=(0, 10), xticks=[0, 5, 10], xlabel="DoS")
    axes[1].axhline(0, color=FERMI, ls="--", zorder=0)
    title(axes[0], "(a) Band structure"); title(axes[1], "(b) DoS")
    fig.suptitle(label, fontsize=16, y=.99)
    legend(fig, handles, labels, right=True)
    save(fig, name)

### S1.9–11 — Projected DoS

In [8]:
handles, labels = BAND_HANDLES, BAND_LABELS
for name, folder, material, segments, ymax in [
    ("S1.9.pdf", "F_Graphene-Borophene_HSE", "Borophene", [(8,16),(0,8)], 4),
    ("S1.10.pdf", "E_Graphene-BC3_HSE", r"BC$_3$", [(8,16),(0,2),(2,8)], 3),
    ("S1.11.pdf", "G_Graphene-B4C3_HSE", r"B$_4$C$_3$", [(7,15),(0,4),(4,7)], 2.5),
]:
    energy, total, projected, fermi = dos(folder, projected_grid=True)
    fig, axes = grid(2, 2, 6.4, right_legend=True)
    titles = ["Total DoS + projections", "Graphene: C", material + ": B", material + ": C"]
    for i, ax in enumerate(axes.flat):
        if i > len(segments): ax.axis("off"); continue
        partial = projected.sum(axis=0) if i == 0 else projected[slice(*segments[i-1])].sum(axis=0)
        sums = total if i == 0 else partial.sum(axis=0)
        for values, color in zip([sums, partial[0], partial[3], partial[1], partial[2]], PDOS_COLORS):
            ax.plot(energy, values, color=color)
        ax.axvline(0, color=FERMI, ls="--", zorder=0)
        ax.set(xlim=(-6,6), ylim=(0,10 if i == 0 else ymax), xticks=np.arange(-6,7,3))
        title(ax, f"({chr(97+i)}) " + titles[i])
        if i % 2 == 0: ax.set_ylabel("Density of states")
        if i >= 2 or (len(segments) == 2 and i == 1): ax.set_xlabel("Energy (eV)")
    pdos_handles = [Line2D([], [], color=c) for c in PDOS_COLORS] + [handles[-1]]
    legend(fig, pdos_handles, ["Total", r"$s$", r"$p_x$", r"$p_y$", r"$p_z$", "Fermi energy"], right=True)
    save(fig, name)

### proj1.8 — Schottky barriers

In [9]:
fig,axes=grid(1,2,5.1)
fig.subplots_adjust(left=.08,right=.98,bottom=.22,top=.95,wspace=.30)
ax=axes[0,0]
ax.set(xlim=(0,1),ylim=(0,1));ax.axis("off")
title(ax,"(a) Schottky barriers")
for y,label,color in [(.76,r"$E_C$",BLUE),(.60,r"$E_F$",FERMI),(.29,r"$E_V$","#EB731E")]:
    ax.plot([.12,.92],[y,y],color=color)
    ax.text(.07,y,label,ha="right",va="center",fontsize=16)
for x,lo,hi,label,color in [(.62,.60,.76,r"$\Phi_n$",BLUE),(.35,.29,.60,r"$\Phi_p$","#EB731E")]:
    ax.annotate("",(x,hi),(x,lo),arrowprops={"arrowstyle":"<->","color":color,"lw":1.5})
    ax.text(x+.04,(lo+hi)/2,label,va="center",fontsize=16)
for y,label in [(.18,r"$\Phi_n<\Phi_p$: n-type SB"),(.09,r"$\Phi_n>\Phi_p$: p-type SB"),
                (0,r"$\Phi_n$ or $\Phi_p\approx0$: Ohmic contact")]:
    ax.text(.04,y,label,fontsize=14,va="center")

source=ROOT/"3_Bandstructure/G_Graphene-B4C3_top_HSE/vaspout.h5"
with h5py.File(source) as f:
    g=f["results/electron_eigenvalues_kpoints_opt"]
    e=g["eigenvalues"][0]-f["results/electron_dos_kpoints_opt/efermi"][()]
    k=g["kpoint_coords"][:]
x=np.r_[0,np.cumsum(np.linalg.norm(np.diff(k,axis=0),axis=1))];length=x[-1];x/=length
ticks=x[[0,len(k)//3-1,2*len(k)//3-1,len(k)-1]]
ax=axes[0,1];ax.plot(x,e,color=VIOLET)
ax.set(xlim=(0,1),ylim=(-3,3),yticks=np.arange(-3,4),ylabel="Energy (eV)",
       xlabel=r"Wave vector ($k$)",xticks=ticks,xticklabels=[r"$\Gamma$","K","M",r"$\Gamma$"])
title(ax,r"(b) Graphene-B$_4$C$_3$ (HSE06)")
for tick in ticks[1:-1]:ax.axvline(tick,color=GREY,ls="--",zorder=0)
ax.axhline(0,color=FERMI,ls="--",zorder=0)
for level,color,start,label in [(1.42,BLUE,ticks[-2],r"$\Phi_n$"),(-.87,"#EB731E",1-.08/length,r"$\Phi_p$")]:
    ax.hlines(level,start,1,colors=color,linestyles="--")
    ax.annotate("",(1-.05/length,level),(1-.05/length,0),
                arrowprops={"arrowstyle":"<->","color":color,"lw":1.5})
    ax.text(1-.2/length,level/2,label,color=color,fontsize=16,va="center")
legend(fig,[Line2D([],[],color=FERMI,ls="--")],["Fermi energy"],columns=1)
save(fig,"proj1.8_schottky.pdf")

### S1.1 — k-point convergence

In [10]:
fig,axes=grid(1,1,4.4)
fig.subplots_adjust(left=.18,bottom=.17,top=.96)
ax=axes[0,0]
lines=(ROOT / "1_Kpoints/Graphene_BC3_Hollow/energy_kpoint.dat").read_text().splitlines()[1:]
xy=np.array([(int(re.search(r"\((\d+),",s)[1]),float(s.split()[-1])) for s in lines])
xy=xy[(xy[:,0]>=9)&(xy[:,0]<=41)]
ax.plot(xy[:,0],xy[:,-1],"o-",color=BLUE,mfc="white")
ax.set(xlabel=r"$k$-point mesh ($X\times X\times1$)",ylabel="Energy (eV)",xticks=xy[::2,0])
ax.yaxis.set_major_formatter(FormatStrFormatter("%.3f"))
title(ax,r"Graphene-BC$_3$ (Hollow)")
save(fig,"S1.1.pdf")

### S1.2 — Monolayer lattice constants

In [11]:
fig,axes=grid(2,2,6.4,right_legend=True)
fig.subplots_adjust(left=.14,wspace=.44)
for i,(ax,folder,label,color) in enumerate(zip(axes.flat,
    ["A_BC3","B_Borophene","C_B4C3","D_Graphene"],
    [r"BC$_3$","Borophene",r"B$_4$C$_3$","Graphene"],[BLUE,GREEN,VIOLET,GREY])):
    xy=table("0_Lattice/"+folder+"/free_energy_lattice.dat")
    x,y=fit_eos(xy[:,0],xy[:,-1])
    ax.plot(x,y,color=color);ax.plot(xy[:,0],xy[:,-1],"o",color=color,mfc="white",ms=5)
    j=np.argmin(y);ax.plot(x[j],y[j],"o",color=color,ms=6)
    title(ax,f"({chr(97+i)}) "+label)
    ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
    ax.xaxis.set_major_locator(MaxNLocator(3))
    if i%2==0: ax.set_ylabel("Energy (eV)")
    if i>=2: ax.set_xlabel(r"Lattice constant ($\mathrm{\AA}$)")
legend(fig,[Line2D([],[],color=GREY),Line2D([],[],color=GREY,marker="o",mfc="white",ls=""),
            Line2D([],[],color=GREY,marker="o",ls="")],
       ["Fitted curve","Source data","Fitted\nminimum"],right=True)
save(fig,"S1.2.pdf")

### S1.3 — Bilayer lattice constants

In [12]:
fig,axes=grid(2,2,6.5)
fig.subplots_adjust(left=.14,bottom=.13,wspace=.44,hspace=.43)
for i,(ax,prefix,label) in enumerate(zip(axes.flat,
    ["E_Graphene-BC3_","F_Graphene-Borophene_","G_Graphene-B4C3_"],
    [r"Graphene-BC$_3$","Graphene-Borophene",r"Graphene-B$_4$C$_3$"])):
    sites=["Top","Bridge","Hollow"] if i==0 else ["Top","Bridge","Hollow1","Hollow2"]
    for site,color in zip(sites,[BLUE,GREEN,VIOLET,"#C82364"]):
        xy=table("0_Lattice/"+prefix+site+"/free_energy_lattice.dat")
        x,y=fit_eos(xy[:,0],xy[:,-1]);ax.plot(x,y,color=color)
        ax.plot(xy[:,0],xy[:,-1],"o",color=color,mfc="white",ms=5)
        j=np.argmin(y);ax.plot(x[j],y[j],"o",color=color,ms=6)
    title(ax,f"({chr(97+i)}) "+label)
    ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
    ax.xaxis.set_major_locator(MaxNLocator(3))
    ax.set_xlabel(r"Lattice constant ($\mathrm{\AA}$)")
    if i%2==0: ax.set_ylabel("Energy (eV)")
axes[1,1].axis("off")
axes[1,1].legend([Line2D([],[],color=c,marker="o",mfc="white") for c in [BLUE,GREEN,VIOLET,"#C82364"]]
    +[Line2D([],[],color=GREY,marker="o",ls="")],
    ["Top","Bridge",r"Hollow (BC$_3$)"+"\nHollow 1 (others)","Hollow 2","Fitted minimum"],loc="center",frameon=True,fancybox=True)
save(fig,"S1.3.pdf")

### S1.4–6 — Lattice constants and layer spacings

In [3]:
# The colour field uses the original linear interpolation. Only the actual
# sampled minimum is marked; the old diagonal scan was not a 2D minimization.
for name,prefix,sites,cmap in [
    ("S1.4.pdf","E_Graphene-BC3_",["Bridge","Hollow","Top"],"Blues_r"),
    ("S1.5.pdf","F_Graphene-Borophene_",["Bridge","Hollow1","Hollow2","Top"],"Greens_r"),
    ("S1.6.pdf","G_Graphene-B4C3_",["Bridge","Hollow1","Hollow2","Top"],"Purples_r"),
]:
    fig,axes=grid(2,2,7.1)
    fig.subplots_adjust(left=.10,right=.86,bottom=.18,wspace=.63,hspace=.35)
    for i,(ax,site) in enumerate(zip(axes.flat,sites)):
        xyz=table("0_Lattice_Distance/"+prefix+site+"/lattice_distance.dat")
        a,d,e=xyz.T
        aa,dd=np.meshgrid(np.linspace(a.min(),a.max(),400),np.linspace(d.min(),d.max(),400))
        ee=griddata((a,d),e,(aa,dd),method="linear")
        cp=ax.pcolormesh(aa,dd,ee,shading="auto",cmap=cmap,alpha=.75,
                         vmax=e.min()+np.ptp(e)*.125,rasterized=True)
        cbar=fig.colorbar(cp,ax=ax,pad=.025,fraction=.045)
        cbar.ax.yaxis.set_major_locator(MaxNLocator(4))
        cbar.ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
        idx=np.argmin(e)
        ax.plot(a[idx],d[idx],"o",mfc="white",mec="black",ms=6,zorder=5)
        title(ax,f"({chr(97+i)}) "+site.replace("Hollow1","Hollow 1").replace("Hollow2","Hollow 2"))
        ax.set_xticks(np.round(a.min() + np.ptp(a) * np.array([.2, .8]), 2))
        ax.yaxis.set_major_locator(MaxNLocator(4))
        ax.xaxis.set_major_formatter(FormatStrFormatter("%.2f"))
        ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
        if i%2==0: ax.set_ylabel(r"Interlayer spacing ($\mathrm{\AA}$)")
        if i>=2: ax.set_xlabel(r"Lattice constant ($\mathrm{\AA}$)")
    if len(sites)==3: axes[1,1].axis("off")
    fig.text(.98,.58,"Energy (eV)",ha="right",va="center",rotation=90,fontsize=16)
    legend(fig,[Line2D([],[],marker="o",color="black",mfc="white",ls="")],
           ["Minimum of sampled data"],columns=1)
    save(fig,name)

### proj1.12–14 — Bilayer optical properties

In [14]:
optical("proj1.12_optics.pdf", [("alpha",r"$\alpha$ (nm$^{-1}$)"),("loss","Energy-loss spectrum")], HSE,[0,2],6.4)
optical("proj1.13_optics.pdf", [("R","Reflectivity"),("n","Refractive index")], HSE,[0,2],6.4)
optical("proj1.14_cor.pdf", [("k","Extinction coefficient")], HSE,[0,2],4.0)

### S1.14 — Monolayer optical properties

In [15]:
optical("S1.14_optics.pdf", [("alpha",r"$\alpha$ (nm$^{-1}$)"),("loss",r"$L$"),
        ("n",r"$n$"),("R",r"$R$"),("k",r"$\kappa$")],MONOLAYERS,[0,1,2],10.0)

### S1.20–24 — Bilayer optical properties in three directions

In [16]:
for name, quantity, ylabel in [("S1.20.pdf","alpha",r"$\alpha$ (nm$^{-1}$)"),
    ("S1.21.pdf","loss","Energy-loss spectrum"),("S1.22.pdf","n","Refractive index"),
    ("S1.23_correct.pdf","R","Reflectivity"),("S1.24_correct.pdf","k","Extinction coefficient")]:
    optical(name,[(quantity,ylabel)],HSE,[0,1,2],4.0)

### S1.13 — Monolayer dielectric functions

In [17]:
# Four monolayers, two tensor directions, one legend for real and imaginary parts.
fig, axes = grid(4,2,10.0)
fig.subplots_adjust(left=.10,bottom=.11,hspace=.20)
for row,(label,source,color) in enumerate(MONOLAYERS):
    energy, epsilon = dielectric(source)
    keep = energy <= 24
    for col,direction in enumerate([0,2]):
        ax=axes[row,col]
        ax.plot(energy[keep],epsilon[direction,direction,keep,0],color=color)
        ax.plot(energy[keep],epsilon[direction,direction,keep,1],color=color,ls="--")
        ax.set(xlim=(0,25),xticks=[0,5,10,15,20,25])
        low,high=ax.get_ylim();ax.set_ylim(low,high+.24*(high-low))
        title(ax,f"({chr(97+row*2+col)}) "+label+": "+NAMES[direction])
        if col == 0: ax.set_ylabel("Dielectric function")
        if row == 3: ax.set_xlabel("Photon energy (eV)")
        else: ax.tick_params(labelbottom=False)
legend(fig,[Line2D([],[],color=GREY),Line2D([],[],color=GREY,ls="--")],
       ["Real part","Imaginary part"],columns=2)
save(fig,"S1.13_dielectric.pdf")

### proj1.9–11 — Bilayer dielectric functions

In [18]:
# Main-text real/imaginary rows preserve the original off-diagonal zoom ranges.
for index,name in [(1,"proj1.9.pdf"),(0,"proj1.10_diff.pdf"),(2,"proj1.11_diff.pdf")]:
    label,prefix,hse,color=BILAYERS[index]
    components=[(0,0),(2,2)]+([(0,1)] if index != 1 else [])
    fig,axes=grid(2,len(components),6.3)
    for row in range(2):
        for col,(i,j) in enumerate(components):
            ax=axes[row,col]
            upper=(2 if index == 0 else 16) if i != j else 24
            for suffix,shade,functional in [("PBE_K65_Normal",color,"PBE"),(hse,ORANGE,"HSE06")]:
                energy,epsilon=dielectric(prefix+"_"+suffix); keep=energy<=upper
                ax.plot(energy[keep],epsilon[i,j,keep,row],color=shade,label=functional)
            ax.set_xlim(0,upper if i != j else 25)
            title(ax,f"({chr(97+row*len(components)+col)}) "+[NAMES[0],NAMES[2],NAMES[3]][col])
            if row == 1: ax.set_xlabel("Photon energy (eV)")
        axes[row,0].set_ylabel(r"Real part, $\varepsilon_1$" if row==0 else r"Imaginary part, $\varepsilon_2$")
    handles,labels=axes[0,0].get_legend_handles_labels()
    legend(fig,handles,labels,columns=2)
    save(fig,name)

### S1.17–19 — Complete dielectric tensors

In [19]:
# Retain all nine raw tensor components, including unequal transposed entries.
for index,name in enumerate(["S1.18.pdf","S1.17_alt.pdf","S1.19.pdf"]):
    label,prefix,hse,color=BILAYERS[index]
    fig,axes=grid(3,3,8.3)
    fig.subplots_adjust(bottom=.20,wspace=.37,hspace=.25)
    for p,(ax,(i,j),heading) in enumerate(zip(axes.flat,COMPONENTS,NAMES)):
        for suffix,shade,functional in [("PBE_K65_Normal",color,"PBE"),(hse,ORANGE,"HSE06")]:
            energy,epsilon=dielectric(prefix+"_"+suffix);keep=energy<=24
            for part,style in [(0,"-"),(1,"--")]:
                ax.plot(energy[keep],epsilon[i,j,keep,part],color=shade,ls=style,
                        label=("Real" if part==0 else "Imaginary")+f" ({functional})")
        ax.set(xlim=(0,25),xticks=[0,10,20])
        if index==1 and i!=j: ax.set_ylim(-.1,.4)
        title(ax,f"({chr(97+p)}) "+heading)
        if p%3==0: ax.set_ylabel("Dielectric function")
        if p>=6: ax.set_xlabel("Photon energy (eV)")
        else: ax.tick_params(labelbottom=False)
    handles,labels=axes[0,0].get_legend_handles_labels()
    legend(fig,handles,labels,columns=2)
    save(fig,name)

### S1.12,15,16 — Optical convergence

In [20]:
# Convergence plots: the original separate energy windows are retained.
for name,sources,labels,colors,windows in [
    ("S1.12_alt.pdf",["D_Graphene_PBE_K33","D_Graphene_PBE_K65","D_Graphene_PBE_K129"],
     [r"$33\times33\times1$",r"$65\times65\times1$",r"$129\times129\times1$"],
     [BLUE,VIOLET,"#C82364"],[(0,5),(10,15)]),
    ("S1.15.pdf",[f"G_Graphene-B4C3_PBE_K17_N{n}" for n in [32,64,128,256,512]],
     [f"{n} bands" for n in [32,64,128,256,512]],
     ["#F03C64",ORANGE,GREEN,BLUE,"#643CC3"],[(0,24),(0,24)]),
    ("S1.16_alt.pdf",["F_Graphene-Borophene_"+s for s in ["PBE_K65_Normal","HSE_K17","HSE_K65_Normal_EDIFF-4","HSE_K65_Normal_EDIFF-5","HSE_K65_Accurate"]],
     ["PBE",r"HSE06, $17\times17\times1$",r"HSE06, EDIFF=$10^{-4}$",r"HSE06, EDIFF=$10^{-5}$",r"HSE06, EDIFF=$10^{-6}$ (Accurate)"],
     ["#AAAFBE",GREEN,BLUE,"#643CC3",ORANGE],[(0,4),(12,15)]),
]:
    fig,axes=grid(2,2,6.7)
    fig.subplots_adjust(bottom=.30,right=.96)
    for row in range(2):
        for col,direction in enumerate([0,2]):
            ax=axes[row,col];low,high=windows[col]
            for source,label,color in zip(sources,labels,colors):
                energy,epsilon=dielectric(source);keep=(energy>=low)&(energy<=high)
                ax.plot(energy[keep],epsilon[direction,direction,keep,row],color=color,label=label)
            ax.set_xlim(low,high)
            ymin,ymax=ax.get_ylim();ax.set_ylim(ymin,ymax+.24*(ymax-ymin))
            title(ax,f"({chr(97+2*row+col)}) "+NAMES[direction])
            if row==1: ax.set_xlabel("Photon energy (eV)")
        axes[row,0].set_ylabel(r"Real part, $\varepsilon_1$" if row==0 else r"Imaginary part, $\varepsilon_2$")
    handles,labels=axes[0,0].get_legend_handles_labels()
    legend(fig,handles,labels,columns=2)
    save(fig,name)

### Save source records and check thesis copies

In [21]:
(HERE / "electronic_sources.json").write_text(json.dumps(electronic.SOURCES, indent=2) + "\n")
(HERE / "optical_sources.json").write_text(json.dumps(optics.SOURCES, indent=2) + "\n")
(HERE / "energy_sources.json").write_text(json.dumps(ENERGY_SOURCES, indent=2) + "\n")
(HERE / "structure_manifest.json").write_text(json.dumps({
    "source_directory": "2_Structure_and_CDD",
    "source_notebooks": ["0.1_structure_figure.ipynb", "2.0_charge_density_differences_figure.ipynb"],
    "changes": "Original raster pixels; new native panel tabs; original bond-marker positions and colours.",
    "files": manifest,
}, indent=2) + "\n")


for name in sorted(GENERATED):
    assert (HERE / name).read_bytes() == (THESIS / name).read_bytes(), name
print(f"{len(GENERATED)} figures saved; all thesis copies match.")